# Collect input data for evapotranspiration modelling

In this notebook, data required to run evapotranspiration modelling is collected from Copernicus Data Space Ecosystem (CDSE) using the openEO interface.

This notebook can be run on the [Copernicus Dataspace Jupyterhub](https://jupyterhub.dataspace.copernicus.eu) in which case data downloads are downloaded since both the data and the execution environment are on CDSE. 

**Note** You should select on of the kernels with GDAL installed, eg. "Geo science".

First we check that Sen-ET Toolbox is installed (and install it if necessary) and then we import all the necessary packages.

In [ ]:
try:
    import senet_toolbox
except ModuleNotFoundError:
    !pip install senet_toolbox@git+https://github.com/DHI/Sen-ET-OpenEO-toolbox.git

In [ ]:
from pathlib import Path
import openeo
from senet_toolbox.utils import date_selector, visualization
from senet_toolbox.utils.general_utils import dump_area_date_info, read_area_date_info
from senet_toolbox.workflows import collect_input_data

## 1. Select Area of Interest
To keep the data organised, and make processing of timeseries easier, the input and output data is kept in Area of Interest (AOI) folders. All data within an AOI folder has the same extent and grid. 

In the cell below, select location where you want to store the data and the AOI name. When running on CDSE Jupyterhub it is recommended to keep it within `./mystorage`, otherwise data will be erased between sessions.

If you are setting up a new AOI and then use cell below to draw a polygon on the map, for the extent you want to process. It is recomended to select AOI's of around 30 km by 30 km so that LST sharpening works well and memory consumption is not too significant. 

If you are working with existing AOI then cell will just visualize the area

In [ ]:
data_dir = "./mystorage/"
aoi_name = "botswana"
aoi_data_dir = Path(data_dir) / aoi_name

In [ ]:
# Draw or visualize AOI extent when setting up a new AOI
map, bboxs = visualization.select_aoi(aoi_data_dir)
map

## 2. Select date from from available Sentinel-3 acquisitions 

**Note**: max_cloud_cover refers to the full tile’s coverage, not just your AOI, so results may vary. It is recomended to also use the [Copernicus Browser](https://browser.dataspace.copernicus.eu) to check both Sentinel-3 and Sentinel-2 data avaiability over the AOI within the specified time extent.

In [ ]:
# Define search parameters
start_date = "2023-05-01"
end_date = "2023-05-10"
max_cloud_cover = 10 # Filter out high-cloud-coverage scenes

# Search for available Sentinel-3 imagery
date_selection = date_selector.select_date(
    aoi_data_dir = aoi_data_dir,
    start_date=start_date,
    end_date=end_date,
    max_cloud_cover=max_cloud_cover
)

## 3. Connect to OpenEO Backend

The Sentinel-3 and Sentinel-2 imagery, Worldcover landcover map, and Copernicus Digital Elevation model are all downloaded from the CDSE OpenEO interface. Run the cell below to authenticate in OpenEO. 

**Note:** you might be required to click an authentication link which appears and follow the instructions.

In [ ]:
connection = openeo.connect("https://openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

## 4. Download Sentinel 2 and Sentinel 3 data for the specified AOI and date

Download Sentinel-3 Land Surface Temperature (LST) and View Zenith Angle (VZA), and Sentinel-2 reflectance data.

When downloading Sentinel-2, keep the `use_biopar_processor=True` setting to also download leaf area index and other biophysical parameters. The `sentinel2_search_range` parameter indicates how many dates before the specifed date to look for Sentinel-2 data. The output file will have the nearest cloud free pixel composite created from the avaiable Sentinel-2 dates. To speed up processing, it is recomended to use [Copernicus Browser](https://browser.dataspace.copernicus.eu) to look through Sentinel-2 imagery on and before the specified date and to set this number as low as possible. 

**Note**: For large areas the download and aggregation of data in OpenEO will take a while and might fail. It is recommended to process smaller regions at the time. 
Go to [https://openeo.dataspace.copernicus.eu/](https://openeo.dataspace.copernicus.eu/) and sign in to follow the jobs and see any errors. 

In [ ]:
s3_lst_path,s3_vza_path,s3_mask_path = collect_input_data.collect_sentinel3_data(
    connection=connection,
    bbox=bboxs[-1],
    date=date_selection.value,
    aoi_name=aoi_name,
    out_dir=data_dir,
)

#### You can now visualize the Sentinel 3 Land Surface Temperature (LST), VZA or Mask

In [ ]:
visualization.show_raster_map(
    s3_lst_path, # change to s3_vza_path,s3_mask_path 
    cmap="inferno"
)

In [ ]:
s2_path = collect_input_data.collect_sentinel2_data(
    connection=connection,
    bbox=bboxs[-1],
    date=date_selection.value,
    aoi_name=aoi_name,
    out_dir=data_dir,
    sentinel2_search_range = 5,
    use_biopar_processor = True
)

Visualize the Sentinel 2 RGB Data to confirm that everything looks fine

In [ ]:
visualization.show_raster_map(
    s2_path,
    rgb=True
)

## 5. Download static AOI data

The Worldcover landcover map and the digital elevation model do not change through time, so they only need to be downloaded once when setting up a new AOI. If you are working with an existing AOI then you can skip the next two cells.

In [ ]:
worldcover_path = collect_input_data.collect_worldcover_data(
    connection=connection,
    bbox=bboxs[-1],
    date=date_selection.value,
    aoi_name=aoi_name,
    s2_template_path=s2_path,
    out_dir=data_dir,
)

Visualize the World Cover data (here we just see the color, each color represent a different class in the Worldcover dataset)

In [ ]:
visualization.show_raster_map(
    worldcover_path,
    cmap="tab20c"
)

In [ ]:
dem_path = collect_input_data.collect_dem_data(
    connection=connection,
    bbox=bboxs[-1],
    date=date_selection.value,
    aoi_name=aoi_name,
    s2_template_path=s2_path,
    out_dir=data_dir,
)

In [ ]:
visualization.show_raster_map(
    dem_path,
    cmap="terrain",
    opacity=0.9
)